In [1]:
import geopandas as gpd
import geemap
import ee

import os
from glob import glob

In [2]:
# List available cities in mnt/
cities = [d for d in os.listdir('mnt') if os.path.isdir(f'mnt/{d}') and d.startswith('20')]
print("Available cities:")

for i, city in enumerate(sorted(cities), 1):
    print(f"{i}. {city}")


Available cities:
1. 2025-02-tunisia-tunis
2. 2025-10-indonesia-sofifi
3. 2025-10-senegal-dakar
4. 2025-10-senegal-diourbel
5. 2025-10-senegal-matam
6. 2025-10-senegal-tambacounda


In [3]:
# Select city
city_dir = sorted(cities)[-1] # Change this to your city

cityname  = city_dir.split('-')[-1] 
data_dir = os.path.join('mnt', city_dir, '02-process-output', 'spatial')
aoi_dir = os.path.join('mnt', city_dir, '01-user-input', 'AOI')
aoi_file = glob(aoi_dir + "/*.shp")[0]

print(f"\n {cityname} - selected: {city_dir}. data_dir: {data_dir}")
print(aoi_file)




 tambacounda - selected: 2025-10-senegal-tambacounda. data_dir: mnt/2025-10-senegal-tambacounda/02-process-output/spatial
mnt/2025-10-senegal-tambacounda/01-user-input/AOI/senegal_tambacounda.shp


In [ ]:
# Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

In [5]:
# ============================================
# 1. LOAD AOI FROM LOCAL SHAPEFILE
# ============================================
aoi_gdf = gpd.read_file(aoi_file)
aoi = geemap.gdf_to_ee(aoi_gdf)

# Define time range
start_year = 2016
end_year = 2025

In [6]:
import numpy as np
ghs_builts = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')\
    .map(lambda img: img.clip(aoi)) 

for year in np.arange(1975, 2035, 5):
    print(year)

    g = ghs_builts.filter(ee.Filter.eq('system:index', str(year)))

    ee.batch.Export.image.toDrive(
        image=g.first(),
        description=f'{cityname}_ghs_built_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f'{cityname}_ghs_built_{year}',
        region=aoi.geometry(),
        # crs='EPSG:4326',
        # scale=100,
        maxPixels=1e13
    ).start()

1975
1980
1985
1990
1995
2000
2005
2010
2015
2020
2025
2030
